<a href="https://colab.research.google.com/github/a-mhamdi/jlai/blob/main/Codes/Julia/Part-3/transfer-learning/transfer-learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning
---

In [1]:
versioninfo()  # -> 1.12.6

Julia Version 1.12.6
Commit 15346901f00 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 2 × Intel(R) Xeon(R) CPU @ 2.00GHz
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, skylake-avx512)
  GC: Built with stock GC
Threads: 2 default, 1 interactive, 2 GC (on 2 virtual cores)
Environment:
  LD_LIBRARY_PATH = /usr/lib64-nvidia
  JULIA_NUM_THREADS = auto


In [2]:
pkgs = """[deps]
CUDA = "052768ef-5323-5732-b1bb-66c8b64840ba"
Flux = "587475ba-b771-5e3f-ad9e-33799f191a9c"
MLDatasets = "eb30cadb-4394-5ae3-aed4-317e484a6458"
Metalhead = "dbeba491-748d-5e0e-a39e-b530a07fa0cc"
ProgressMeter = "92933f4c-e287-5a05-a399-4b506db050ca"
Statistics = "10745b16-79ce-11e8-11f9-7d13ad32a3b2"
cuDNN = "02a925ec-e4fe-4b08-9a7e-0d78e3d38ccd"
"""

open("Project.toml", "w") do file
    write(file, pkgs)
end

356

In [3]:
_ = begin
  import Pkg;
  Pkg.activate(".");
  Pkg.instantiate();
end

  Activating project at `/content`


In [4]:
Pkg.status()

Status `/content/Project.toml`
  [052768ef] CUDA v6.5.1
  [587475ba] Flux v0.16.15
  [eb30cadb] MLDatasets v0.7.22
  [dbeba491] Metalhead v0.9.8
  [92933f4c] ProgressMeter v1.13.0
  [10745b16] Statistics v1.11.5
  [02a925ec] cuDNN v6.5.1


Load the pre-trained model

In [5]:
using Metalhead

In [6]:
resnet = ResNet(18; pretrain=true)

ResNet(
  Chain(
    Chain(
      Chain(
        Conv((7, 7), 3 => 64, pad=3, stride=2, bias=false),  # 9_408 parameters
        BatchNorm(64, relu),            # 128 parameters, plus 128
        MaxPool((3, 3), pad=1, stride=2),
      ),
      Chain(
        Parallel(
          addact(relu, ...),
          identity,
          Chain(
            Conv((3, 3), 64 => 64, pad=1, bias=false),  # 36_864 parameters
            BatchNorm(64),              # 128 parameters, plus 128
            NNlib.relu,
            Conv((3, 3), 64 => 64, pad=1, bias=false),  # 36_864 parameters
            BatchNorm(64),              # 128 parameters, plus 128
          ),
        ),
        Parallel(
          addact(relu, ...),
          identity,
          Chain(
            Conv((3, 3), 64 => 64, pad=1, bias=false),  # 36_864 parameters
            BatchNorm(64),              # 128 parameters, plus 128
            NNlib.relu,
            Conv((3, 3), 64 => 64, pad=1, bias=false),  # 36_864 parameters
   

[API Reference](https://fluxml.ai/Metalhead.jl/dev/api/reference/#API-Reference)

In [7]:
using Flux
using Flux: onecold, onehotbatch

In [8]:
using CUDA

In [9]:
model = Chain(
    resnet.layers[1:end-1],
    resnet.layers[end][1:end-1],
    # Replace the last layer
    Dense(512, 10)
    ) |> gpu

Chain(
  Chain(
    Chain(
      Chain(
        Conv((7, 7), 3 => 64, pad=3, stride=2, bias=false),  # 9_408 parameters
        BatchNorm(64, relu),            # 128 parameters, plus 128
        MaxPool((3, 3), pad=1, stride=2),
      ),
      Chain(
        Parallel(
          addact(relu, ...),
          identity,
          Chain(
            Conv((3, 3), 64 => 64, pad=1, bias=false),  # 36_864 parameters
            BatchNorm(64),              # 128 parameters, plus 128
            NNlib.relu,
            Conv((3, 3), 64 => 64, pad=1, bias=false),  # 36_864 parameters
            BatchNorm(64),              # 128 parameters, plus 128
          ),
        ),
        Parallel(
          addact(relu, ...),
          identity,
          Chain(
            Conv((3, 3), 64 => 64, pad=1, bias=false),  # 36_864 parameters
            BatchNorm(64),              # 128 parameters, plus 128
            NNlib.relu,
            Conv((3, 3), 64 => 64, pad=1, bias=false),  # 36_864 parameters
    

Load the MNIST dataset

In [10]:
using MLDatasets

In [11]:
function get_data(split; batchsize=64)
    d = MNIST(split)
    X = reshape(d.features, 28, 28, 1, :)
    y = onehotbatch(d.targets, 0:9)
    loader = Flux.DataLoader((X, y); batchsize, shuffle=(split == :train))
    return loader
end

get_data (generic function with 1 method)

In [12]:
train_loader = get_data(:train);
test_loader = get_data(:test);

**Define a loss function and an optimizer**

In [13]:
loss_fn = Flux.logitcrossentropy
opt_state = Flux.setup(Adam(3e-3), model[end]) # Freeze the weights of the pre-trained layers

(weight = Leaf(Adam(eta=0.003, beta=(0.9, 0.999), epsilon=1.0e-8), (CuArray(Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]), CuArray(Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]), (0.9, 0.999))), bias = Leaf(Adam(eta=0.003, beta=(0.9, 0.999), epsilon=1.0e-8), (CuArray(Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), CuArray(Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), (0.9, 0.999))), σ = ())

In [14]:
# Per-batch preprocessing
const IMAGENET_MEAN = reshape(Float32[0.485, 0.456, 0.406], 1, 1, 3, 1)
const IMAGENET_STD  = reshape(Float32[0.229, 0.224, 0.225], 1, 1, 3, 1)

1×1×3×1 Array{Float32, 4}:
[:, :, 1, 1] =
 0.229

[:, :, 2, 1] =
 0.224

[:, :, 3, 1] =
 0.225

In [15]:
function to_resnet_input(X; size=(64, 64))
    X = repeat(X; outer=(1, 1, 3, 1))                # 28×28×1×B -> 28×28×3×B
    X = Flux.upsample_bilinear(X; size=size)         # 28×28 -> 64×64 (or 224×224)
    return (X .- IMAGENET_MEAN) ./ IMAGENET_STD      # ImageNet normalization
end

to_resnet_input (generic function with 1 method)

Fine-tune the model

In [16]:
using ProgressMeter, Statistics

In [17]:
for epoch in 1:5
    @info "TRAINING..."
    Flux.trainmode!(model)
    p = Progress(length(train_loader); desc="Epoch $epoch train: ", showspeed=true)

    for (X, y) in train_loader
        X = X |> to_resnet_input
        X, y = (X, y) |> gpu

        local ŷ
        loss, ∇ = Flux.withgradient(model) do m
            ŷ = m(X)
            loss_fn(ŷ, y)
        end
        Flux.update!(opt_state, model.layers[end], ∇[1].layers[end])

        acc = mean(Flux.onecold(ŷ) .== Flux.onecold(y))
        next!(p; showvalues = [
            (:train_loss, round(loss; digits=4)),
            (:train_acc,  round(100 * acc; digits=2)),
        ])
    end
    finish!(p)

    @info "TESTING..."
    Flux.testmode!(model)
    p = Progress(length(test_loader); desc="Epoch $epoch test:  ", showspeed=true)
    tot_loss, correct, total = 0f0, 0, 0

    for (X, y) in test_loader
        X = X |> to_resnet_input
        X, y = (X, y) |> gpu
        ŷ = model(X)

        n = size(y, 2)
        tot_loss += loss_fn(ŷ, y) * n
        correct  += sum(Flux.onecold(ŷ) .== Flux.onecold(y))
        total    += n

        next!(p; showvalues = [
            (:test_loss, round(tot_loss / total; digits=4)),
            (:test_acc,  round(100 * correct / total; digits=2)),
        ])
    end
    finish!(p)
end

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/OPrdc/src/ProgressMeter.jl:651
Epoch 5 test:  100%|██████████████████████| Time: 0:00:04 (27.35 ms/it)
   test_loss: 0.4019
    test_acc: 87.96
